In [2]:
numbers = (x * 2 for x in range(1_000_000))   # ~200 bytes in RAM
next(numbers)

0

In [10]:
next(numbers)

16

In [16]:
def count_up(n):
    print("Generator started")
    for i in range(n):
        print(f"  About to yield {i}")
        yield i
        print(f"  Resumed after yielding {i}")
    print("*************** Generator done")

gen = count_up(3)
print("Generator object created — nothing executed yet!")
print(type(gen))  # 
print()

# Each call to next() advances to the next yield
print("Calling next() #1:", next(gen))
print()
print("Calling next() #2:", next(gen))
print()
print("Calling next() #3:", next(gen))

import sys

N = 1_000_000
list_mem   = sys.getsizeof(list(range(N)))
gen_mem    = sys.getsizeof(x for x in range(N))
print(f"List size:      {list_mem:,} bytes  ({list_mem // 1024 // 1024} MB)")
print(f"Generator size: {gen_mem} bytes")
print(f"Memory ratio:   {list_mem // gen_mem:,}x more RAM for the list")

Generator object created — nothing executed yet!
<class 'generator'>

Generator started
  About to yield 0
Calling next() #1: 0

  Resumed after yielding 0
  About to yield 1
Calling next() #2: 1

  Resumed after yielding 1
  About to yield 2
Calling next() #3: 2
List size:      8,000,056 bytes  (7 MB)
Generator size: 192 bytes
Memory ratio:   41,666x more RAM for the list


In [17]:

# Full anatomy: a generator that parses transaction records
def parse_transactions(raw_lines):
    """Yields one parsed transaction dict at a time — never loads all into RAM."""
    for line in raw_lines:
        line = line.strip()
        if not line or line.startswith('#'):
            continue  # Skip blank lines and comments
        parts = line.split(',')
        yield {
            'id':     parts[0],
            'amount': float(parts[1]),
            'type':   parts[2]
        }


# Simulated raw file (in production, this would be open('file.csv'))
raw = [
    "# Transaction log",
    "TXN001,150.00,CREDIT",
    "TXN002,-45.50,DEBIT",
    "",
    "TXN003,2000.00,CREDIT",
]

# Iterate: each transaction is processed one at a time
for tx in parse_transactions(raw):
    print(f"  Processed: {tx}")

  Processed: {'id': 'TXN001', 'amount': 150.0, 'type': 'CREDIT'}
  Processed: {'id': 'TXN002', 'amount': -45.5, 'type': 'DEBIT'}
  Processed: {'id': 'TXN003', 'amount': 2000.0, 'type': 'CREDIT'}


In [19]:
import sys

data = range(100_000)

# Approach 1: List Comprehension — materializes everything NOW
squares_list = [x ** 2 for x in data]
print(f"List:      {sys.getsizeof(squares_list):,} bytes — ALL values in RAM")

# Approach 2: Generator Expression — computes on demand
squares_gen = (x ** 2 for x in data)
print(f"Generator: {sys.getsizeof(squares_gen)} bytes — no values computed yet")

print()

# Both work identically in for loops
total_gen  = sum(x ** 2 for x in range(10))   # Generator: O(1) memory
total_list = sum([x ** 2 for x in range(10)])  # List: O(n) memory
print(f"Both produce same result: {total_gen} == {total_list}: {total_gen == total_list}")

print()

# ✅ PREFER generator form for large sum/max/min/any/all operations
huge_data = range(10_000_000)
result = sum(x for x in huge_data if x % 2 == 0)  # No list ever created
print(f"Sum of even numbers 0-10M: {result:,}")



List:      800,984 bytes — ALL values in RAM
Generator: 200 bytes — no values computed yet

Both produce same result: 285 == 285: True

Sum of even numbers 0-10M: 24,999,995,000,000


In [20]:
import io

# Simulate a multi-GB file with a StringIO stream
def make_fake_file(n_rows):
    """Creates a fake CSV in memory to simulate a large file."""
    lines = ["txn_id,amount,currency,status"]
    for i in range(n_rows):
        lines.append(f"T{i:06d},{(i % 5000) + 0.99:.2f},USD,{'SETTLED' if i % 3 else 'PENDING'}")
    return io.StringIO('\n'.join(lines))


# ✅ Generator-based pipeline — processes row by row
def read_transactions(file_obj):
    """Yields one raw row at a time from the file."""
    next(file_obj)  # Skip header
    for line in file_obj:
        yield line.strip()

def parse_transactions(raw_rows):
    """Parses each raw row into a structured dict."""
    for row in raw_rows:
        parts = row.split(',')
        yield {'id': parts[0], 'amount': float(parts[1]),
               'currency': parts[2], 'status': parts[3]}

def filter_large(transactions, threshold=1000.0):
    """Passes through only high-value transactions."""
    for tx in transactions:
        if tx['amount'] >= threshold:
            yield tx


# Build pipeline — nothing executes until we iterate
fake_file    = make_fake_file(n_rows=50_000)
raw_rows     = read_transactions(fake_file)
parsed       = parse_transactions(raw_rows)
high_value   = filter_large(parsed, threshold=4000.0)

# Drive the pipeline — processes 50K rows with O(1) memory
results = list(high_value)
print(f"Processed 50,000 rows — found {len(results)} high-value transactions")
print(f"Sample: {results[0]}")

Processed 50,000 rows — found 10000 high-value transactions
Sample: {'id': 'T004000', 'amount': 4000.99, 'currency': 'USD', 'status': 'SETTLED'}


In [21]:



def even_numbers(data):
    for i in data:
        if i % 2 == 0:
            yield i
result = list(even_numbers(range(21)))
print (f" These are the even numbers in range 21 {result}")

 These are the even numbers in range 21 [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20]


In [24]:
from collections.abc import Generator
from typing import Any

# EXERCISE 2: Write a generator pipeline that:
# 1. Reads raw strings like "Alice,85" (name,score)
# 2. Parses them into dicts
# 3. Filters to scores >= 80
# All as separate generators — no intermediate lists


raw = ["Alice,85", "Bob,72", "Charlie,91", "Diana,78", "Eve,88"]

def parse_scores(raw_lines)-> Generator[dict[str, int | str], None, None]:
    for line in raw_lines:
        name, score = line.split(',')
        yield {'name': name, 'score': int(score) }
        
def filter_passing(records, threshold = 80)-> Generator[dict[str, Any], None, None]:
    for record in records:
        if record["score"] >= threshold:
            yield record

parsed = parse_scores(raw)
passing = filter_passing(parsed)

for student in passing:
    print(f"  {student}")    


  {'name': 'Alice', 'score': 85}
  {'name': 'Charlie', 'score': 91}
  {'name': 'Eve', 'score': 88}


In [25]:
# EXERCISE 3: Write a generator that produces a running cumulative sum
# For input [1, 2, 3, 4, 5] it should yield 1, 3, 6, 10, 15
# Bonus: make it work on infinite sequences (don't load all into memory)

def running_total(numbers):
    total = 0
    for n in numbers:
        total += n
        yield total
result = list(running_total([1, 2, 3, 4, 5]))
print(f"Exercise 3a: {result}")
# Expected: [1, 3, 6, 10, 15]


Exercise 3a: [1, 3, 6, 10, 15]


In [27]:
def infinite_fibonacci():
    a, b = 0, 1
    while True:       # ⬅️ An infinite loop!
        yield a       # It pauses here before the loop explodes
        a, b = b, a + b
fib = infinite_fibonacci()


0


In [34]:
print(next(fib))

13


In [36]:
larger_number = next(x for x in infinite_fibonacci() if x > 1000)
print(larger_number)

1597


In [42]:
# A generator that reads fake transaction strings: "name,amount"
def transaction_reader(lines):
    for line in lines:
        name, amount = line.split(',')
        yield {'name': name, 'amount': int(amount)}

# Our raw data
raw_data = ["Alice,50", "Bob,150", "Charlie,90"]

# 1. We create the generator 
parsed_txns = transaction_reader(raw_data)

# 2. We filter using a Generator Expression
high_value = (tx for tx in parsed_txns if tx['amount'] > 100)
print(high_value)
print(next(high_value) )
#high_value_list = list(high_value)   #Generator Object
#print(high_value_list)    #Actual List

<generator object <genexpr> at 0x000001844373DF20>
{'name': 'Bob', 'amount': 150}


In [43]:
# Setup: DuckDB — zero config, runs in-process, full SQL
# pip install duckdb
import duckdb

# Create an in-memory connection
con = duckdb.connect()

# -------------------------------------------------------
# Seed data: Citi-style server telemetry (employees for ranking demos)
# -------------------------------------------------------
con.execute("""
CREATE TABLE employees AS
SELECT * FROM (VALUES
    ('Alice',   'Engineering', 120000),
    ('Bob',     'Engineering', 120000),
    ('Charlie', 'Engineering', 110000),
    ('Diana',   'Engineering',  95000),
    ('Eve',     'Marketing',   105000),
    ('Frank',   'Marketing',   105000),
    ('Grace',   'Marketing',    88000),
    ('Henry',   'Finance',     115000),
    ('Ivy',     'Finance',     115000),
    ('Jack',    'Finance',      92000)
) t(name, department, salary)
""")

con.execute("""
CREATE TABLE server_metrics AS
SELECT * FROM (VALUES
    ('server-01', DATE '2026-01-01', 55.0),
    ('server-01', DATE '2026-01-02', 62.0),
    ('server-01', DATE '2026-01-03', 58.0),
    ('server-01', DATE '2026-01-04', 71.0),
    ('server-01', DATE '2026-01-05', 89.0),
    ('server-01', DATE '2026-01-06', 94.0),
    ('server-01', DATE '2026-01-07', 88.0),
    ('server-02', DATE '2026-01-01', 40.0),
    ('server-02', DATE '2026-01-02', 45.0),
    ('server-02', DATE '2026-01-03', 43.0),
    ('server-02', DATE '2026-01-04', 50.0),
    ('server-02', DATE '2026-01-05', 48.0),
    ('server-02', DATE '2026-01-06', 52.0),
    ('server-02', DATE '2026-01-07', 55.0)
) t(server_id, collection_date, cpu_utilization)
""")

print("✅ Tables created: employees, server_metrics")
con.execute("SELECT * FROM employees").df()

✅ Tables created: employees, server_metrics


,name,department,salary
0,Alice,Engineering,120000
1,Bob,Engineering,120000
2,Charlie,Engineering,110000
3,Diana,Engineering,95000
4,Eve,Marketing,105000
5,Frank,Marketing,105000
6,Grace,Marketing,88000
7,Henry,Finance,115000
8,Ivy,Finance,115000
9,Jack,Finance,92000


In [44]:
print(
    con.execute(""" 

SELECT 
    name, 
    department, 
    salary,
    -- Here is the magic:
    AVG(salary) OVER (PARTITION BY department) as dept_average
FROM employees
"""
).df() )

      name   department  salary   dept_average
0    Alice  Engineering  120000  111250.000000
1      Bob  Engineering  120000  111250.000000
2  Charlie  Engineering  110000  111250.000000
3    Diana  Engineering   95000  111250.000000
4    Henry      Finance  115000  107333.333333
5      Ivy      Finance  115000  107333.333333
6     Jack      Finance   92000  107333.333333
7      Eve    Marketing  105000   99333.333333
8    Frank    Marketing  105000   99333.333333
9    Grace    Marketing   88000   99333.333333


In [45]:
print(
    con.execute(""" 

SELECT 
    name, 
    department, 
    salary,
    DENSE_RANK() OVER (PARTITION BY department ORDER BY salary DESC) as rank
FROM employees
"""
).df() )

      name   department  salary  rank
0    Alice  Engineering  120000     1
1      Bob  Engineering  120000     1
2  Charlie  Engineering  110000     2
3    Diana  Engineering   95000     3
4    Henry      Finance  115000     1
5      Ivy      Finance  115000     1
6     Jack      Finance   92000     2
7      Eve    Marketing  105000     1
8    Frank    Marketing  105000     1
9    Grace    Marketing   88000     2


In [46]:
print(
    con.execute(""" 

SELECT 
    name, 
    department, 
    salary,
    RANK() OVER (PARTITION BY department ORDER BY salary DESC) as rank
FROM employees
"""
).df() )

      name   department  salary  rank
0    Alice  Engineering  120000     1
1      Bob  Engineering  120000     1
2  Charlie  Engineering  110000     3
3    Diana  Engineering   95000     4
4      Eve    Marketing  105000     1
5    Frank    Marketing  105000     1
6    Grace    Marketing   88000     3
7    Henry      Finance  115000     1
8      Ivy      Finance  115000     1
9     Jack      Finance   92000     3


In [50]:
print(
    con.execute(""" 

With ranked_employees as (
SELECT 
    name, 
    department, 
    salary,
    RANK() OVER (PARTITION BY department ORDER BY salary DESC) as rank
FROM employees ) select name, department, salary, rank from ranked_employees where rank <= 3
"""
).df() )

      name   department  salary  rank
0    Henry      Finance  115000     1
1      Ivy      Finance  115000     1
2     Jack      Finance   92000     3
3      Eve    Marketing  105000     1
4    Frank    Marketing  105000     1
5    Grace    Marketing   88000     3
6    Alice  Engineering  120000     1
7      Bob  Engineering  120000     1
8  Charlie  Engineering  110000     3


In [54]:
print(
    con.execute(""" 

With ranked_employees as (
SELECT 
    name, 
    department, 
    salary,
    -- 1. The Rank
    DENSE_RANK() OVER (PARTITION BY department ORDER BY salary DESC) as dept_rank,
    
    -- 2. The Average
    ROUND(AVG(salary) OVER (PARTITION BY department), 0) as dept_avg,
    
    -- 3. The Maximum (Who makes the most here?)
    MAX(salary) OVER (PARTITION BY department) as dept_max,
    
    -- 4. The Distance from the Average (Math using a window function!)
    salary - ROUND(AVG(salary) OVER (PARTITION BY department), 0) as diff_from_avg
    
FROM employees ) select name, department, salary,dept_avg, dept_max,diff_from_avg, dept_rank from ranked_employees where dept_rank <= 3
"""
).df() )

      name   department  salary  dept_avg  dept_max  diff_from_avg  dept_rank
0      Eve    Marketing  105000   99333.0    105000         5667.0          1
1    Frank    Marketing  105000   99333.0    105000         5667.0          1
2    Grace    Marketing   88000   99333.0    105000       -11333.0          2
3    Alice  Engineering  120000  111250.0    120000         8750.0          1
4      Bob  Engineering  120000  111250.0    120000         8750.0          1
5  Charlie  Engineering  110000  111250.0    120000        -1250.0          2
6    Diana  Engineering   95000  111250.0    120000       -16250.0          3
7    Henry      Finance  115000  107333.0    115000         7667.0          1
8      Ivy      Finance  115000  107333.0    115000         7667.0          1
9     Jack      Finance   92000  107333.0    115000       -15333.0          2


In [56]:

print(
    con.execute(""" 

SELECT
    server_id,
    collection_date,
    cpu_utilization as today_cpu,
    
    -- Look backwards 1 row to see yesterday!
    LAG(cpu_utilization, 1) OVER (
        PARTITION BY server_id 
        ORDER BY collection_date
    ) AS yesterday_cpu
    
FROM server_metrics

"""
).df() )


    server_id collection_date  today_cpu  yesterday_cpu
0   server-01      2026-01-01       55.0            NaN
1   server-01      2026-01-02       62.0           55.0
2   server-01      2026-01-03       58.0           62.0
3   server-01      2026-01-04       71.0           58.0
4   server-01      2026-01-05       89.0           71.0
5   server-01      2026-01-06       94.0           89.0
6   server-01      2026-01-07       88.0           94.0
7   server-02      2026-01-01       40.0            NaN
8   server-02      2026-01-02       45.0           40.0
9   server-02      2026-01-03       43.0           45.0
10  server-02      2026-01-04       50.0           43.0
11  server-02      2026-01-05       48.0           50.0
12  server-02      2026-01-06       52.0           48.0
13  server-02      2026-01-07       55.0           52.0


In [58]:
print(
    con.execute(""" 

SELECT
    server_id,
    collection_date,
    cpu_utilization AS today_cpu,
    
    -- This defines the "logic" for the calculation
    LAG(cpu_utilization, 1) OVER (
        PARTITION BY server_id 
        ORDER BY collection_date
    ) AS yesterday_cpu
    
FROM server_metrics

-- This defines the "presentation" for your eyes
ORDER BY server_id, collection_date;

"""
                
).df() )



    server_id collection_date  today_cpu  yesterday_cpu
0   server-01      2026-01-01       55.0            NaN
1   server-01      2026-01-02       62.0           55.0
2   server-01      2026-01-03       58.0           62.0
3   server-01      2026-01-04       71.0           58.0
4   server-01      2026-01-05       89.0           71.0
5   server-01      2026-01-06       94.0           89.0
6   server-01      2026-01-07       88.0           94.0
7   server-02      2026-01-01       40.0            NaN
8   server-02      2026-01-02       45.0           40.0
9   server-02      2026-01-03       43.0           45.0
10  server-02      2026-01-04       50.0           43.0
11  server-02      2026-01-05       48.0           50.0
12  server-02      2026-01-06       52.0           48.0
13  server-02      2026-01-07       55.0           52.0


In [61]:
print(
    con.execute(""" 
with daily_deltas as (
SELECT
    server_id,
    collection_date,
    cpu_utilization AS today_cpu,
    
    -- This defines the "logic" for the calculation
    cpu_utilization - LAG(cpu_utilization, 1) OVER (
        PARTITION BY server_id 
        ORDER BY collection_date
    ) AS CPU_SPIKE
    
FROM server_metrics

-- This defines the "presentation" for your eyes
ORDER BY server_id, collection_date
) select * from daily_deltas
where CPU_SPIKE > 10

"""
                
).df() )


   server_id collection_date  today_cpu  CPU_SPIKE
0  server-01      2026-01-04       71.0       13.0
1  server-01      2026-01-05       89.0       18.0


In [64]:
print(
    con.execute(""" 

SELECT 
    server_id,
    collection_date,
    cpu_utilization,
    AVG(cpu_utilization) OVER (
        PARTITION BY server_id
        ORDER BY collection_date
        -- The Magic Frame:
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) as rolling_7d_avg
    
FROM server_metrics

"""
                
).df() )

    server_id collection_date  cpu_utilization  rolling_7d_avg
0   server-01      2026-01-01             55.0       55.000000
1   server-01      2026-01-02             62.0       58.500000
2   server-01      2026-01-03             58.0       58.333333
3   server-01      2026-01-04             71.0       61.500000
4   server-01      2026-01-05             89.0       67.000000
5   server-01      2026-01-06             94.0       71.500000
6   server-01      2026-01-07             88.0       73.857143
7   server-02      2026-01-01             40.0       40.000000
8   server-02      2026-01-02             45.0       42.500000
9   server-02      2026-01-03             43.0       42.666667
10  server-02      2026-01-04             50.0       44.500000
11  server-02      2026-01-05             48.0       45.200000
12  server-02      2026-01-06             52.0       46.333333
13  server-02      2026-01-07             55.0       47.571429


In [65]:
import duckdb
# 1. Create an in-memory database connection
con = duckdb.connect()
# 2. Create the table and insert the fake daily revenue data
con.execute("""
CREATE TABLE store_sales AS
SELECT * FROM (VALUES
    ('Store-A', DATE '2026-03-01', 500),
    ('Store-A', DATE '2026-03-02', 200),
    ('Store-A', DATE '2026-03-03', 700),
    ('Store-B', DATE '2026-03-01', 100),
    ('Store-B', DATE '2026-03-02', 300),
    ('Store-B', DATE '2026-03-03', 150)
) t(store_id, sale_date, revenue)
""")
# 3. Run the Cumulative Revenue Window Function Query
result = con.execute("""
SELECT 
    store_id,
    sale_date,
    revenue as daily_revenue,
    
    -- The Magic!
    SUM(revenue) OVER (
        PARTITION BY store_id
        ORDER BY sale_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) as cumulative_revenue
    
FROM store_sales
ORDER BY store_id, sale_date
""").df()  # .df() turns it into a nice pandas DataFrame
# 4. Print the result
print(result.to_string(index=False))

store_id  sale_date  daily_revenue  cumulative_revenue
 Store-A 2026-03-01            500               500.0
 Store-A 2026-03-02            200               700.0
 Store-A 2026-03-03            700              1400.0
 Store-B 2026-03-01            100               100.0
 Store-B 2026-03-02            300               400.0
 Store-B 2026-03-03            150               550.0


In [68]:
# 3. Run the Cumulative Revenue Window Function Query
result = con.execute("""
SELECT 
    store_id,
    sale_date,
    revenue as daily_revenue,
    
    -- The Magic!
    SUM(revenue) OVER (
        PARTITION BY store_id
        ORDER BY sale_date
        ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING 
    ) as cumulative_revenue
    
FROM store_sales
ORDER BY store_id, sale_date
""").df()  # .df() turns it into a nice pandas DataFrame
# 4. Print the result
print(result.to_string(index=False))

store_id  sale_date  daily_revenue  cumulative_revenue
 Store-A 2026-03-01            500              1400.0
 Store-A 2026-03-02            200               900.0
 Store-A 2026-03-03            700               700.0
 Store-B 2026-03-01            100               550.0
 Store-B 2026-03-02            300               450.0
 Store-B 2026-03-03            150               150.0


In [69]:
# 3. Run the Cumulative Revenue Window Function Query
result = con.execute("""
SELECT 
    store_id,
    sale_date,
    revenue as daily_revenue,
    
    -- The Magic!
    SUM(revenue) OVER (
        PARTITION BY store_id
        ORDER BY sale_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) as cumulative_revenue
    
FROM store_sales
ORDER BY store_id, sale_date
""").df()  # .df() turns it into a nice pandas DataFrame
# 4. Print the result
print(result.to_string(index=False))

store_id  sale_date  daily_revenue  cumulative_revenue
 Store-A 2026-03-01            500              1400.0
 Store-A 2026-03-02            200              1400.0
 Store-A 2026-03-03            700              1400.0
 Store-B 2026-03-01            100               550.0
 Store-B 2026-03-02            300               550.0
 Store-B 2026-03-03            150               550.0


In [ ]:



SELECT 
    name, 
    department, 
    salary,
    -- No partition! It just counts every row.
    ROW_NUMBER() OVER (ORDER BY salary DESC) as global_rank
FROM employees


